# Multi-product validation of SWOT water surface elevation in the Pantanal Rivers

This notebook is linked to the manuscript:

**Multi-product validation of SWOT water surface elevation in the Pantanal Rivers**

Jahdy Moreno-Oliveira a,b*, Alice Fassoni-Andrade b, Mark A. Trigg c, Daniel Medeiros Moreira d, Evlyn Marcia Leão de Moraes Novoa

a National Institute for Space Research (INPE), São Jose dos Campos, Brazil  
b Institute of Geosciences, University of Brasília (UnB), Brasília, Brazil  
c School of Civil Engineering, University of Leeds, Leeds, United Kingdom  
d Geological Survey of Brazil, Rio de Janeiro, Brazil

The notebook implements the main methodological workflow used for the article: in situ data harmonization, SWOT product time-series preparation, DAHITI/HydroWeb consolidation, quality control, Hampel outlier filtering, temporal matching, absolute validation, dynamic validation, and final figure/table export.

This notebook can run as a standalone file: if the companion Python module is missing, the bootstrap cell recreates it in the current working directory.


In [ ]:
# Optional dependency installation.
# Uncomment and run this cell only if the environment does not already have the required libraries.

# %pip install numpy pandas matplotlib scipy scikit-learn geopandas shapely pyproj pyogrio fiona xarray h5netcdf rioxarray rasterio earthaccess requests contextily py-hydroweb


In [ ]:
# Standalone bootstrap.
# If this notebook is copied without the companion Python module, this cell recreates
# the module in the current working directory so the notebook can run by itself.
from pathlib import Path

module_path = Path("swot_wse_pipeline.py")
if not module_path.exists():
    module_path.write_text('from __future__ import annotations\n\nimport json\nimport os\nimport re\nimport shutil\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport scipy.stats as stats\n\n\nPRODUCTS_SWOT = ["pixc", "raster", "node", "reach"]\nPRODUCTS_ALL = PRODUCTS_SWOT + ["altimetry"]\nPRODUCT_LABELS = {\n    "pixc": "PIXC",\n    "raster": "Raster",\n    "node": "Node",\n    "reach": "Reach",\n    "altimetry": "Altimetry",\n}\nPRODUCT_COLORS = {\n    "pixc": "#e67e22",\n    "raster": "#2e7d32",\n    "node": "#c0392b",\n    "reach": "#6c5ce7",\n    "altimetry": "#6f6f6f",\n}\nHYDRO_PHASE_ORDER = ["Low", "Rising", "High", "Falling"]\n\n\n@dataclass\nclass PipelineConfig:\n    project_root: Path = Path("..").resolve()\n    in_situ_unleveled_dir: Path = Path("data/in_situ/unleveled")\n    in_situ_gnss_leveled_dir: Path = Path("data/in_situ/gnss_leveled")\n    metadata_file: Path = Path("data/metadata/station_metadata.csv")\n    water_mask_file: Path = Path("data/masks/water_mask.shp")\n    stations_vector_file: Path = Path("data/vectors/gauge_locations.gpkg")\n    altimetry_codes_file: Path = Path("data/altimetry/altimetry_station_codes.csv")\n    swot_station_dir_template: str = "outputs/swot_timeseries/Station_{station}"\n    altimetry_station_dir_template: str = "outputs/altimetry_timeseries/Station_{station}"\n    tables_dir: Path = Path("outputs/tables")\n    figures_dir: Path = Path("outputs/figures")\n    date_start: str = "2023-03-31"\n    date_end: str = "2025-08-01"\n    calval_cutoff_date: str = "2023-07-01"\n    match_window_days: int = 3\n    station_filter: tuple[str, ...] = tuple()\n    hydroweb_bbox: tuple[float, float, float, float] = (\n        -59.283966,\n        -22.1041297,\n        -54.9274659,\n        -16.033635,\n    )\n    run_swot_downloads: bool = False\n    run_altimetry_downloads: bool = False\n    overwrite_outputs: bool = False\n\n\ndef paths(cfg: PipelineConfig) -> dict[str, Path]:\n    root = Path(cfg.project_root).resolve()\n\n    p = {\n        "root": root,\n        "in_situ_unleveled": root / cfg.in_situ_unleveled_dir,\n        "in_situ_gnss_leveled": root / cfg.in_situ_gnss_leveled_dir,\n        "metadata": root / cfg.metadata_file,\n        "water_mask": root / cfg.water_mask_file,\n        "stations_vector": root / cfg.stations_vector_file,\n        "altimetry_codes": root / cfg.altimetry_codes_file,\n        "tables": root / cfg.tables_dir,\n        "figures": root / cfg.figures_dir,\n    }\n    for key in ["tables", "figures"]:\n        p[key].mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef station_dir(cfg: PipelineConfig, template: str, station: str) -> Path:\n    return Path(cfg.project_root).resolve() / template.format(station=normalize_station_id(station))\n\n\ndef config_as_json(cfg: PipelineConfig) -> str:\n    return json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2, ensure_ascii=False)\n\n\ndef validate_inputs(cfg: PipelineConfig, strict: bool = False) -> pd.DataFrame:\n    p = paths(cfg)\n    required = [\n        "metadata",\n        "water_mask",\n        "stations_vector",\n        "altimetry_codes",\n        "in_situ_unleveled",\n        "in_situ_gnss_leveled",\n    ]\n    rows = [{"item": name, "path": str(p[name]), "exists": p[name].exists()} for name in required]\n    report = pd.DataFrame(rows)\n    if strict and not report["exists"].all():\n        missing = report.loc[~report["exists"], ["item", "path"]].to_dict("records")\n        raise FileNotFoundError(f"Missing required inputs: {missing}")\n    return report\n\n\ndef normalize_station_id(value) -> str:\n    if pd.isna(value):\n        return ""\n    text = str(value).replace("\\ufeff", "").replace("\\xa0", " ").strip()\n    text = re.sub(r"\\s+", "", text)\n    if re.fullmatch(r"\\d+\\.0+", text):\n        return text.split(".")[0]\n    if re.fullmatch(r"\\d+(?:\\.\\d+)?", text):\n        try:\n            number = float(text)\n            if number.is_integer():\n                return str(int(number))\n        except Exception:\n            pass\n    return text\n\n\ndef safe_numeric(values):\n    if isinstance(values, pd.Series):\n        text = values.astype(str).str.strip().str.replace("%", "", regex=False).str.replace(" ", "", regex=False)\n        direct = pd.to_numeric(values, errors="coerce")\n        decimal_comma = pd.to_numeric(text.str.replace(",", ".", regex=False), errors="coerce")\n        thousand_dot = pd.to_numeric(\n            text.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),\n            errors="coerce",\n        )\n        return direct.combine_first(decimal_comma).combine_first(thousand_dot)\n    try:\n        return float(str(values).replace(",", "."))\n    except Exception:\n        return np.nan\n\n\ndef read_csv_robust(path: Path | str, dtype=None) -> pd.DataFrame:\n    path = Path(path)\n    attempts = [\n        {"sep": None, "engine": "python", "encoding": "utf-8-sig", "on_bad_lines": "skip"},\n        {"sep": ",", "engine": "python", "encoding": "utf-8-sig", "on_bad_lines": "skip"},\n        {"sep": ";", "engine": "python", "encoding": "utf-8-sig", "on_bad_lines": "skip"},\n        {"sep": "\\t", "engine": "python", "encoding": "utf-8-sig", "on_bad_lines": "skip"},\n    ]\n    best = None\n    last_error = None\n    for kwargs in attempts:\n        try:\n            df = pd.read_csv(path, dtype=dtype, **kwargs)\n            df.columns = [str(c).replace("\\ufeff", "").strip() for c in df.columns]\n            if best is None or len(df.columns) > len(best.columns):\n                best = df\n            if len(df.columns) > 1:\n                return df\n        except Exception as exc:\n            last_error = exc\n    if best is not None:\n        return best\n    raise last_error or ValueError(f"Could not read CSV: {path}")\n\n\ndef parse_datetime_from_candidates(df: pd.DataFrame, candidates: Iterable[str]) -> tuple[pd.Series, str | None]:\n    best = pd.Series(pd.NaT, index=df.index)\n    best_score = -1\n    best_col = None\n    cols = list(dict.fromkeys(\n        [c for c in candidates if c in df.columns]\n        + [c for c in df.columns if any(k in str(c).lower() for k in ["date", "time", "data"])]\n    ))\n    for col in cols:\n        raw = df[col]\n        options = [pd.to_datetime(raw, errors="coerce", utc=True)]\n        numeric = pd.to_numeric(raw, errors="coerce")\n        if numeric.notna().any():\n            for unit in ["s", "ms", "us", "ns"]:\n                options.append(pd.to_datetime(numeric, unit=unit, errors="coerce", utc=True))\n        for parsed in options:\n            if getattr(parsed.dt, "tz", None) is not None:\n                parsed = parsed.dt.tz_localize(None)\n            years = parsed.dropna().dt.year\n            realistic = int(((years >= 1990) & (years <= 2100)).sum())\n            score = realistic * 100000 + int(parsed.notna().sum())\n            if score > best_score:\n                best = parsed\n                best_score = score\n                best_col = col\n    return best, best_col\n\n\ndef normalize_hydro_phase(value):\n    if pd.isna(value):\n        return np.nan\n    text = str(value).strip().lower()\n    mapping = {\n        "low": "Low",\n        "seca": "Low",\n        "dry": "Low",\n        "rising": "Rising",\n        "enchente": "Rising",\n        "subida": "Rising",\n        "high": "High",\n        "cheia": "High",\n        "flood": "High",\n        "falling": "Falling",\n        "vazante": "Falling",\n        "receding": "Falling",\n        "descida": "Falling",\n    }\n    return mapping.get(text, value if value in HYDRO_PHASE_ORDER else np.nan)\n\n\ndef infer_hydro_phase(wse: pd.Series) -> pd.Series:\n    series = safe_numeric(pd.Series(wse)).interpolate(limit_direction="both")\n    if len(series) < 4 or series.nunique(dropna=True) < 3:\n        return pd.Series(np.nan, index=series.index)\n    q25, q75 = series.quantile([0.25, 0.75])\n    slope = series.diff().rolling(3, center=True, min_periods=1).mean()\n    out = pd.Series("Low", index=series.index, dtype=object)\n    out[series >= q75] = "High"\n    out[(series > q25) & (series < q75) & (slope >= 0)] = "Rising"\n    out[(series > q25) & (series < q75) & (slope < 0)] = "Falling"\n    return out\n\n\ndef get_orbital_phase(date, cfg: PipelineConfig):\n    dt = pd.to_datetime(date, errors="coerce")\n    if pd.isna(dt):\n        return np.nan\n    return "CalVal" if dt < pd.Timestamp(cfg.calval_cutoff_date) else "Nominal"\n\n\ndef load_metadata(cfg: PipelineConfig) -> pd.DataFrame:\n    p = paths(cfg)\n    if not p["metadata"].exists():\n        return pd.DataFrame()\n    raw = read_csv_robust(p["metadata"], dtype=str)\n    norm = {c: re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_") for c in raw.columns}\n    df = raw.rename(columns=norm).copy()\n    station_col = next((c for c in ["codigo", "cod_ana", "codigo_ana", "station", "id"] if c in df.columns), None)\n    if station_col is None:\n        raise ValueError("Metadata must include codigo/cod_ana/codigo_ana/station/id.")\n    out = pd.DataFrame({"station": df[station_col].map(normalize_station_id)})\n    aliases = {\n        "station_name": ["nome", "name", "station_name"],\n        "river": ["rio", "river", "river_name"],\n        "latitude": ["latitude", "lat"],\n        "longitude": ["longitude", "lon", "long"],\n        "width": ["largura", "width"],\n        "drainage_area": ["area_drenagem", "drainage_area"],\n        "subregion": ["subregiao", "subregion"],\n        "pantanal": ["pantanal"],\n    }\n    for target, choices in aliases.items():\n        col = next((c for c in choices if c in df.columns), None)\n        out[target] = df[col] if col else np.nan\n    for col in ["latitude", "longitude", "width", "drainage_area"]:\n        out[col] = safe_numeric(out[col])\n    out = out.drop_duplicates("station").query("station != \'\'").reset_index(drop=True)\n    if cfg.station_filter:\n        selected = set(map(normalize_station_id, cfg.station_filter))\n        out = out[out["station"].isin(selected)].reset_index(drop=True)\n    return out\n\n\ndef detect_leveled_stations(cfg: PipelineConfig) -> list[str]:\n    p = paths(cfg)\n    stations = {\n        normalize_station_id(fp.name.replace("_nivelada.csv", "").replace("_leveled.csv", ""))\n        for pattern in ["*_nivelada.csv", "*_leveled.csv"]\n        for fp in p["in_situ_gnss_leveled"].glob(pattern)\n    }\n    if cfg.station_filter:\n        stations &= set(map(normalize_station_id, cfg.station_filter))\n    return sorted(stations)\n\n\ndef load_insitu_series(cfg: PipelineConfig, station: str, leveled: bool = False) -> pd.DataFrame:\n    p = paths(cfg)\n    station = normalize_station_id(station)\n    if leveled:\n        candidates = [\n            p["in_situ_gnss_leveled"] / f"{station}_leveled.csv",\n            p["in_situ_gnss_leveled"] / f"{station}_nivelada.csv",\n        ]\n    else:\n        candidates = [p["in_situ_unleveled"] / f"{station}.csv"]\n    file_path = next((candidate for candidate in candidates if candidate.exists()), candidates[0])\n    if not file_path.exists():\n        return pd.DataFrame()\n    df = read_csv_robust(file_path)\n    df["data"], _ = parse_datetime_from_candidates(df, ["data", "date", "datetime", "data_ref"])\n    wse_col = next((c for c in ["wse_swot_ref", "wse_ana", "wse_egm08", "wse", "WSE", "Cota", "cota", "nivel", "stage"] if c in df.columns), None)\n    if wse_col is None:\n        raise ValueError(f"{file_path}: WSE column not found.")\n    out = pd.DataFrame({"data": df["data"], "wse_insitu": safe_numeric(df[wse_col])})\n    phase_col = next((c for c in ["hydro_phase", "phase", "fase", "fase_hidrologica"] if c in df.columns), None)\n    out["hydro_phase"] = df[phase_col].apply(normalize_hydro_phase) if phase_col else infer_hydro_phase(out["wse_insitu"])\n    out["station"] = station\n    out["leveled"] = bool(leveled)\n    out = out.dropna(subset=["data", "wse_insitu"])\n    out = out[(out["data"] >= pd.Timestamp(cfg.date_start)) & (out["data"] <= pd.Timestamp(cfg.date_end))]\n    return out.sort_values("data").reset_index(drop=True)\n\n\ndef split_codes(value) -> list[str]:\n    if pd.isna(value):\n        return []\n    return [c.strip() for c in str(value).split(",") if c.strip() and c.strip() != "-"]\n\n\ndef standardize_altimetry_df(df: pd.DataFrame, source: str, source_code: str | None = None) -> pd.DataFrame:\n    df = df.copy()\n    df.columns = [str(c).strip() for c in df.columns]\n    date_col = next((c for c in df.columns if any(k in c.lower() for k in ["date", "time", "data", "datetime"])), None)\n    wse_col = next((c for c in df.columns if any(k in c.lower() for k in ["water_level", "wse", "height", "nivel", "level"])), None)\n    unc_col = next((c for c in df.columns if any(k in c.lower() for k in ["wse_u", "uncert", "sigma", "std", "error"])), None)\n    if date_col is None or wse_col is None:\n        raise ValueError(f"Could not detect date/WSE columns for {source}:{source_code}")\n    out = pd.DataFrame(\n        {\n            "date": pd.to_datetime(df[date_col], errors="coerce").dt.normalize(),\n            "wse": safe_numeric(df[wse_col]),\n            "source_virtual_station": f"{source}:{source_code}" if source_code else source,\n        }\n    )\n    if unc_col:\n        out["wse_u"] = safe_numeric(df[unc_col])\n    return out.dropna(subset=["date", "wse"]).sort_values("date")\n\n\ndef download_dahiti_station(dahiti_id: str, out_csv: Path) -> Path:\n    import requests\n\n    api_key = os.getenv("DAHITI_API_KEY", "").strip()\n    if not api_key:\n        raise RuntimeError("Set DAHITI_API_KEY to download DAHITI data.")\n    url = "https://dahiti.dgfi.tum.de/api/v2/download-water-level/"\n    payload = {"api_key": api_key, "dahiti_id": int(dahiti_id), "format": "csv"}\n    response = requests.post(url, json=payload, timeout=60)\n    if response.status_code != 200:\n        response = requests.get(url, params=payload, timeout=60)\n    response.raise_for_status()\n    if response.text.lstrip().startswith("<"):\n        raise RuntimeError(f"Unexpected HTML response for DAHITI {dahiti_id}")\n    out_csv.parent.mkdir(parents=True, exist_ok=True)\n    out_csv.write_text(response.text, encoding="utf-8")\n    return out_csv\n\n\ndef read_hydroweb_txt(path: Path, code: str) -> pd.DataFrame:\n    df = pd.read_csv(\n        path,\n        sep=r"\\s+",\n        comment="#",\n        header=None,\n        usecols=[0, 2, 3],\n        names=["date", "wse", "wse_u"],\n        engine="python",\n    )\n    return standardize_altimetry_df(df, "HYDROWEB", code)\n\n\ndef filename_matches_code(filename: str, code: str) -> bool:\n    return bool(re.search(rf"(?<![A-Za-z0-9]){re.escape(str(code))}(?![A-Za-z0-9])", str(filename)))\n\n\ndef download_hydroweb_zip(cfg: PipelineConfig, out_zip: Path) -> Path:\n    api_key = os.getenv("HYDROWEB_API_KEY", "").strip()\n    if not api_key:\n        raise RuntimeError("Set HYDROWEB_API_KEY to download HydroWeb data.")\n    try:\n        from py_hydroweb import Client, DownloadBasket\n    except Exception as exc:\n        raise ImportError("Install py-hydroweb or place HydroWeb TXT files manually.") from exc\n    out_zip.parent.mkdir(parents=True, exist_ok=True)\n    client = Client(hydroweb_api_url="https://hydroweb.next.theia-land.fr/api", api_key=api_key)\n    basket = DownloadBasket("Pantanal_WSE_Basket")\n    basket.add_collection("HYDROWEB_RIVERS_OPE", bbox=list(cfg.hydroweb_bbox))\n    basket.add_collection("HYDROWEB_RIVERS_RESEARCH", bbox=list(cfg.hydroweb_bbox))\n    client.submit_and_download_zip(basket, zip_filename=out_zip.name, output_folder=out_zip.parent)\n    if not out_zip.exists():\n        raise FileNotFoundError(f"HydroWeb ZIP was not created: {out_zip}")\n    return out_zip\n\n\ndef extract_hydroweb_code_from_zip(cfg: PipelineConfig, code: str, target_txt: Path, zip_path: Path | None = None) -> Path:\n    target_txt.parent.mkdir(parents=True, exist_ok=True)\n    zip_path = zip_path or Path(cfg.project_root).resolve() / "outputs" / "hydroweb_download" / "hydroweb_pantanal.zip"\n    if not zip_path.exists():\n        download_hydroweb_zip(cfg, zip_path)\n    with zipfile.ZipFile(zip_path, "r") as zf:\n        candidates = [name for name in zf.namelist() if name.lower().endswith(".txt") and filename_matches_code(Path(name).name, code)]\n        if not candidates:\n            raise FileNotFoundError(f"HydroWeb code {code} not found in {zip_path}")\n        with zf.open(candidates[0]) as src, target_txt.open("wb") as dst:\n            shutil.copyfileobj(src, dst)\n    return target_txt\n\n\ndef consolidate_altimetry(cfg: PipelineConfig, data_cut: str = "2023-01-01") -> pd.DataFrame:\n    p = paths(cfg)\n    if not p["altimetry_codes"].exists():\n        return pd.DataFrame()\n    mapping = read_csv_robust(p["altimetry_codes"], dtype=str)\n    if "cod_ana" not in mapping.columns:\n        raise ValueError("The altimetry station-code table must include cod_ana.")\n    summary = []\n    for _, row in mapping.iterrows():\n        station = normalize_station_id(row.get("cod_ana"))\n        station_path = station_dir(cfg, cfg.altimetry_station_dir_template, station)\n        series = []\n        for code in split_codes(row.get("cod_dahiti")):\n            raw_csv = station_path / "dahiti" / f"{code}.csv"\n            if cfg.run_altimetry_downloads and (cfg.overwrite_outputs or not raw_csv.exists()):\n                download_dahiti_station(code, raw_csv)\n            if raw_csv.exists():\n                series.append(standardize_altimetry_df(read_csv_robust(raw_csv), "DAHITI", code))\n        for code in split_codes(row.get("cod_hidroweb")):\n            raw_txt = station_path / "hydroweb" / f"{code}.txt"\n            raw_csv = station_path / "hydroweb" / f"{code}.csv"\n            if cfg.run_altimetry_downloads and (cfg.overwrite_outputs or not raw_txt.exists()):\n                try:\n                    extract_hydroweb_code_from_zip(cfg, code, raw_txt)\n                except Exception as exc:\n                    print(f"HydroWeb download {station}/{code}: {exc}")\n            if raw_csv.exists():\n                series.append(standardize_altimetry_df(read_csv_robust(raw_csv), "HYDROWEB", code))\n            elif raw_txt.exists():\n                series.append(read_hydroweb_txt(raw_txt, code))\n        if not series:\n            summary.append({"station": station, "selected_source": None, "n_obs": 0})\n            continue\n        combined = pd.concat(series, ignore_index=True)\n        combined = combined[combined["date"] >= pd.Timestamp(data_cut)].copy()\n        if combined.empty:\n            summary.append({"station": station, "selected_source": None, "n_obs": 0})\n            continue\n        combined["source_name"] = combined["source_virtual_station"].str.split(":").str[0]\n        selected_source = combined.groupby("source_name").size().sort_values(ascending=False).index[0]\n        selected = combined[combined["source_name"] == selected_source].copy()\n        agg = {"wse": "mean", "source_virtual_station": lambda x: " | ".join(sorted(set(map(str, x))))}\n        if "wse_u" in selected.columns:\n            agg["wse_u"] = "mean"\n        final = selected.groupby("date", as_index=False).agg(agg).sort_values("date")\n        final["source_selected"] = selected_source\n        out_csv = station_path / f"alt_conv_{station}.csv"\n        out_csv.parent.mkdir(parents=True, exist_ok=True)\n        final.to_csv(out_csv, index=False)\n        summary.append({"station": station, "selected_source": selected_source, "n_obs": len(final), "output": str(out_csv)})\n    out = pd.DataFrame(summary)\n    out.to_csv(p["tables"] / "altimetry_consolidation_summary.csv", index=False)\n    return out\n\n\ndef calc_metrics(y_true, y_pred, include_bias: bool = True) -> dict:\n    y1 = safe_numeric(pd.Series(y_true))\n    y2 = safe_numeric(pd.Series(y_pred))\n    mask = ~(y1.isna() | y2.isna())\n    y1 = y1[mask].to_numpy(dtype=float)\n    y2 = y2[mask].to_numpy(dtype=float)\n    out = {"mae": np.nan, "rmse": np.nan, "spearman_r": np.nan, "spearman_p": np.nan, "n_obs": int(len(y1))}\n    if include_bias:\n        out["bias"] = np.nan\n    if len(y1) < 2:\n        return out\n    out["mae"] = float(np.mean(np.abs(y2 - y1)))\n    out["rmse"] = float(np.sqrt(np.mean((y2 - y1) ** 2)))\n    rho, p_value = stats.spearmanr(y1, y2)\n    out["spearman_r"] = float(rho) if pd.notna(rho) else np.nan\n    out["spearman_p"] = float(p_value) if pd.notna(p_value) else np.nan\n    if include_bias:\n        out["bias"] = float(np.mean(y2 - y1))\n    return out\n\n\ndef rolling_hampel_flags(values, window_size=5, n_sigma=3.0, min_periods=3, mad_scale=1.4826):\n    series = pd.Series(values, dtype="float64")\n    median = series.rolling(window_size, center=True, min_periods=min_periods).median()\n    mad = (series - median).abs().rolling(window_size, center=True, min_periods=min_periods).median()\n    threshold = n_sigma * mad_scale * mad\n    deviation = (series - median).abs()\n    flags = ((threshold > 0) & (deviation > threshold)).fillna(False)\n    return flags.to_numpy(), median.to_numpy(), mad.to_numpy(), threshold.to_numpy(), deviation.to_numpy()\n\n\ndef apply_hampel_filter(df: pd.DataFrame, value_col="wse_sat", date_col="data", group_cols=("station", "product")):\n    if df.empty or value_col not in df.columns:\n        return df.copy(), df.copy()\n    filtered_parts = []\n    audit_parts = []\n    by = [c for c in group_cols if c in df.columns]\n    groups = df.groupby(by) if by else [(None, df)]\n    for _, group in groups:\n        work = group.sort_values(date_col).copy()\n        work[value_col] = safe_numeric(work[value_col])\n        flags, median, mad, threshold, deviation = rolling_hampel_flags(work[value_col])\n        work["hampel_median"] = median\n        work["hampel_mad"] = mad\n        work["hampel_threshold"] = threshold\n        work["hampel_abs_deviation"] = deviation\n        work["is_hampel_outlier"] = flags\n        work["is_outlier_any"] = flags\n        audit_parts.append(work)\n        filtered_parts.append(work.loc[~work["is_outlier_any"]].copy())\n    filtered = pd.concat(filtered_parts, ignore_index=True) if filtered_parts else pd.DataFrame()\n    audit = pd.concat(audit_parts, ignore_index=True) if audit_parts else pd.DataFrame()\n    return filtered, audit\n\n\ndef nearest_match_no_reuse(df_ref: pd.DataFrame, df_sat: pd.DataFrame, window_days: int) -> pd.DataFrame:\n    if df_ref.empty or df_sat.empty:\n        return pd.DataFrame()\n    ref = df_ref.dropna(subset=["data"]).sort_values("data").reset_index(drop=True).copy()\n    sat = df_sat.dropna(subset=["data"]).sort_values("data").reset_index(drop=True).copy()\n    if ref.empty or sat.empty:\n        return pd.DataFrame()\n    sat_times = sat["data"].astype("int64").to_numpy()\n    ref_times = ref["data"].astype("int64").to_numpy()\n    window_ns = pd.Timedelta(days=window_days).value\n    used = np.zeros(len(sat), dtype=bool)\n    rows = []\n    for i, ref_time in enumerate(ref_times):\n        lo = np.searchsorted(sat_times, ref_time - window_ns, side="left")\n        hi = np.searchsorted(sat_times, ref_time + window_ns, side="right")\n        candidates = np.flatnonzero(~used[lo:hi]) + lo\n        if candidates.size == 0:\n            continue\n        best_idx = candidates[np.abs(sat_times[candidates] - ref_time).argmin()]\n        used[best_idx] = True\n        best = sat.loc[best_idx]\n        row = {\n            "date_ref": ref.loc[i, "data"],\n            "date_sat": best["data"],\n            "wse_insitu": ref.loc[i, "wse_insitu"],\n            "wse_sat": best["wse_sat"],\n            "hydro_phase": ref.loc[i, "hydro_phase"] if "hydro_phase" in ref.columns else np.nan,\n            "orbital_phase": best.get("orbital_phase", np.nan),\n        }\n        for col in sat.columns:\n            if col not in {"data", "wse_sat", "orbital_phase"}:\n                row[col] = best[col]\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\nSWOT_SCHEMAS = {\n    "pixc": {"file": "pixc_complete_raw.csv", "date_cols": ["data", "arquivo"], "wse_cols": ["wse_media", "wse", "height_media"]},\n    "raster": {"file": "raster_complete_raw.csv", "date_cols": ["data", "arquivo"], "wse_cols": ["wse_media", "wse", "height"]},\n    "node": {"file": "node_complete_raw.csv", "date_cols": ["data", "time_str", "timestamp_arquivo", "arquivo_origem"], "wse_cols": ["wse", "WSE", "wse_sm"]},\n    "reach": {"file": "reach_complete_raw.csv", "date_cols": ["data", "time_str", "timestamp_arquivo", "arquivo_origem"], "wse_cols": ["wse", "WSE"]},\n}\n\n\ndef find_product_file(cfg: PipelineConfig, station: str, product: str) -> Path | None:\n    base = station_dir(cfg, cfg.swot_station_dir_template, station)\n    direct = base / product / SWOT_SCHEMAS[product]["file"]\n    if direct.exists():\n        return direct\n    flat = base / SWOT_SCHEMAS[product]["file"]\n    return flat if flat.exists() else None\n\n\ndef load_swot_product(cfg: PipelineConfig, station: str, product: str) -> pd.DataFrame:\n    file_path = find_product_file(cfg, station, product)\n    if file_path is None:\n        return pd.DataFrame()\n    df = read_csv_robust(file_path)\n    schema = SWOT_SCHEMAS[product]\n    df["data"], date_source = parse_datetime_from_candidates(df, schema["date_cols"])\n    wse_col = next((c for c in schema["wse_cols"] if c in df.columns), None)\n    if wse_col is None:\n        return pd.DataFrame()\n    out = df.copy()\n    out["station"] = station\n    out["product"] = product\n    out["wse_sat"] = safe_numeric(df[wse_col])\n    out["orbital_phase"] = out["data"].apply(lambda x: get_orbital_phase(x, cfg))\n    version_source = pd.Series("", index=out.index, dtype=object)\n    for candidate_col in ["arquivo", "arquivo_origem", "filename", "granule"]:\n        if candidate_col in out.columns:\n            version_source = version_source.where(version_source.astype(str).str.len() > 0, out[candidate_col].astype(str))\n    out["swot_version"] = version_source.astype(str).str.extract(r"_(C|D|2\\.0|PIC0|PGC0)_", expand=False)\n    out["time_source_column"] = date_source\n    out["wse_source_column"] = wse_col\n    out = out.dropna(subset=["data", "wse_sat"])\n    out = out[(out["data"] >= pd.Timestamp(cfg.date_start)) & (out["data"] <= pd.Timestamp(cfg.date_end))]\n    out = out[out["wse_sat"] > -1e9]\n    return out.sort_values("data").reset_index(drop=True)\n\n\ndef load_altimetry_product(cfg: PipelineConfig, station: str) -> pd.DataFrame:\n    base = station_dir(cfg, cfg.altimetry_station_dir_template, station)\n    files = list(base.glob("alt_conv_*.csv"))\n    if not files:\n        return pd.DataFrame()\n    parts = []\n    for file_path in files:\n        df = read_csv_robust(file_path)\n        df["data"], _ = parse_datetime_from_candidates(df, ["date", "data"])\n        wse_col = next((c for c in ["wse", "WSE", "water_level", "height"] if c in df.columns), None)\n        if wse_col:\n            parts.append(pd.DataFrame({"data": df["data"], "wse_sat": safe_numeric(df[wse_col])}))\n    if not parts:\n        return pd.DataFrame()\n    out = pd.concat(parts, ignore_index=True).dropna(subset=["data", "wse_sat"])\n    out["data"] = out["data"].dt.normalize()\n    out = out.groupby("data", as_index=False)["wse_sat"].mean()\n    out["station"] = station\n    out["product"] = "altimetry"\n    out["orbital_phase"] = out["data"].apply(lambda x: get_orbital_phase(x, cfg))\n    return out.sort_values("data").reset_index(drop=True)\n\n\ndef quality_filter_product(df: pd.DataFrame, product: str) -> pd.DataFrame:\n    if df.empty:\n        return df\n    out = df.copy()\n    if "swot_version" in out.columns and product in PRODUCTS_SWOT:\n        out = out[~out["swot_version"].astype(str).str.contains("PIC0|PGC0", na=False)]\n    if product == "node" and "node_q" in out.columns:\n        out = out[safe_numeric(out["node_q"]) < 2]\n    if product == "reach" and "reach_q" in out.columns:\n        out = out[safe_numeric(out["reach_q"]) < 2]\n    if product == "raster" and "wse_qual_media" in out.columns:\n        out = out[safe_numeric(out["wse_qual_media"]) < 1.8]\n    return out\n\n\ndef build_matchups(cfg: PipelineConfig, metadata: pd.DataFrame, leveled_stations: list[str]):\n    stage_rows = {"raw": [], "quality_filtered": [], "outlier_filtered": []}\n    audits = []\n    if metadata.empty:\n        return {key: pd.DataFrame() for key in stage_rows}, pd.DataFrame()\n    for _, station_meta in metadata.iterrows():\n        station = station_meta["station"]\n        is_leveled = station in leveled_stations\n        insitu = load_insitu_series(cfg, station, leveled=is_leveled)\n        if insitu.empty:\n            insitu = load_insitu_series(cfg, station, leveled=False)\n        if insitu.empty:\n            continue\n        for product in PRODUCTS_ALL:\n            sat = load_altimetry_product(cfg, station) if product == "altimetry" else load_swot_product(cfg, station, product)\n            if sat.empty:\n                continue\n            raw = sat.copy()\n            quality = quality_filter_product(raw, product)\n            filtered, audit = apply_hampel_filter(quality)\n            if not audit.empty:\n                audits.append(audit.assign(stage="hampel", station=station, product=product))\n            for stage, table in [("raw", raw), ("quality_filtered", quality), ("outlier_filtered", filtered)]:\n                matchups = nearest_match_no_reuse(insitu, table, cfg.match_window_days)\n                if matchups.empty:\n                    continue\n                for col in ["station_name", "river", "latitude", "longitude", "width", "drainage_area", "subregion", "pantanal"]:\n                    matchups[col] = station_meta.get(col, np.nan)\n                matchups["station"] = station\n                matchups["product"] = product\n                matchups["station_group"] = "leveled" if is_leveled else "nonleveled"\n                matchups["lineage_stage"] = stage\n                matchups["signed_error"] = safe_numeric(matchups["wse_sat"]) - safe_numeric(matchups["wse_insitu"])\n                matchups["abs_error"] = matchups["signed_error"].abs()\n                stage_rows[stage].append(matchups)\n    matchups_by_stage = {key: pd.concat(value, ignore_index=True) if value else pd.DataFrame() for key, value in stage_rows.items()}\n    audit_df = pd.concat(audits, ignore_index=True) if audits else pd.DataFrame()\n    return matchups_by_stage, audit_df\n\n\ndef aggregate_metrics(df: pd.DataFrame, keys: list[str], ref_col="wse_insitu", sat_col="wse_sat") -> pd.DataFrame:\n    if df.empty:\n        return pd.DataFrame()\n    rows = []\n    for values, group in df.groupby(keys):\n        values = values if isinstance(values, tuple) else (values,)\n        rec = dict(zip(keys, values))\n        rec.update(calc_metrics(group[ref_col], group[sat_col], include_bias=True))\n        rec["n_stations"] = int(group["station"].nunique()) if "station" in group.columns else np.nan\n        rows.append(rec)\n    return pd.DataFrame(rows)\n\n\ndef build_dynamic_metrics(df: pd.DataFrame):\n    rows = []\n    examples = []\n    if df.empty:\n        return pd.DataFrame(), pd.DataFrame()\n    for (station, product), group in df.groupby(["station", "product"]):\n        group = group.sort_values("date_ref").copy()\n        if len(group) < 5:\n            continue\n        ref = safe_numeric(group["wse_insitu"])\n        sat = safe_numeric(group["wse_sat"])\n        valid = ~(ref.isna() | sat.isna())\n        group = group.loc[valid].copy()\n        if len(group) < 5:\n            continue\n        if group["wse_insitu"].max() == group["wse_insitu"].min() or group["wse_sat"].max() == group["wse_sat"].min():\n            continue\n        group["insitu_norm"] = (group["wse_insitu"] - group["wse_insitu"].min()) / (group["wse_insitu"].max() - group["wse_insitu"].min())\n        group["sat_norm"] = (group["wse_sat"] - group["wse_sat"].min()) / (group["wse_sat"].max() - group["wse_sat"].min())\n        met = calc_metrics(group["insitu_norm"], group["sat_norm"], include_bias=False)\n        rows.append(\n            {\n                "station": station,\n                "product": product,\n                "station_group": group["station_group"].iloc[0],\n                "n_obs": met["n_obs"],\n                "spearman_r": met["spearman_r"],\n                "spearman_p": met["spearman_p"],\n            }\n        )\n        examples.append(group[["station", "product", "date_ref", "insitu_norm", "sat_norm", "station_group", "hydro_phase"]].copy())\n    return pd.DataFrame(rows), pd.concat(examples, ignore_index=True) if examples else pd.DataFrame()\n\n\ndef save_table(df: pd.DataFrame, path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False)\n\n\ndef run_analysis(cfg: PipelineConfig) -> dict[str, pd.DataFrame]:\n    p = paths(cfg)\n    metadata = load_metadata(cfg)\n    leveled = detect_leveled_stations(cfg)\n    matchups, outlier_audit = build_matchups(cfg, metadata, leveled)\n    for stage, df in matchups.items():\n        save_table(df, p["tables"] / f"matchups_{stage}.csv")\n    save_table(outlier_audit, p["tables"] / "outlier_filter_audit.csv")\n\n    all_outlier = matchups.get("outlier_filtered", pd.DataFrame())\n    absolute = all_outlier[all_outlier.get("station_group", pd.Series(dtype=str)).eq("leveled")].copy() if not all_outlier.empty else pd.DataFrame()\n    metrics_overall = aggregate_metrics(absolute, ["product"])\n    metrics_station = aggregate_metrics(absolute, ["station", "product"])\n    metrics_orbital = aggregate_metrics(absolute, ["product", "orbital_phase"]) if not absolute.empty else pd.DataFrame()\n    metrics_phase = aggregate_metrics(absolute[absolute["product"].isin(PRODUCTS_SWOT)] if not absolute.empty else absolute, ["product", "hydro_phase"])\n    metrics_dynamic_station, normalized_examples = build_dynamic_metrics(all_outlier)\n    metrics_dynamic_overall = (\n        metrics_dynamic_station.groupby("product", as_index=False).agg(\n            n_station_series=("station", "size"),\n            n_obs=("n_obs", "sum"),\n            median_spearman=("spearman_r", "median"),\n            mean_spearman=("spearman_r", "mean"),\n        )\n        if not metrics_dynamic_station.empty\n        else pd.DataFrame()\n    )\n\n    tables = {\n        "metadata": metadata,\n        "matchups_raw": matchups.get("raw", pd.DataFrame()),\n        "matchups_quality_filtered": matchups.get("quality_filtered", pd.DataFrame()),\n        "matchups_outlier_filtered": all_outlier,\n        "outlier_filter_audit": outlier_audit,\n        "metrics_absolute_overall": metrics_overall,\n        "metrics_absolute_by_station": metrics_station,\n        "metrics_absolute_by_orbital_phase": metrics_orbital,\n        "metrics_absolute_by_hydro_phase": metrics_phase,\n        "metrics_dynamic_by_station": metrics_dynamic_station,\n        "metrics_dynamic_overall": metrics_dynamic_overall,\n        "normalized_series_for_examples": normalized_examples,\n    }\n\n    output_names = {\n        "metrics_absolute_overall": ["metrics_absolute_overall.csv", "metrics_overall.csv"],\n        "metrics_absolute_by_station": ["metrics_absolute_by_station.csv", "metrics_by_station.csv"],\n        "metrics_absolute_by_orbital_phase": ["metrics_absolute_by_orbital_phase.csv", "metrics_by_product_orbital_phase.csv"],\n        "metrics_absolute_by_hydro_phase": ["metrics_absolute_by_hydro_phase.csv", "error_by_hydrological_phase.csv"],\n        "metrics_dynamic_by_station": ["metrics_dynamic_by_station.csv"],\n        "metrics_dynamic_overall": ["metrics_dynamic_overall.csv"],\n        "normalized_series_for_examples": ["normalized_series_for_examples.csv"],\n    }\n    for key, filenames in output_names.items():\n        for filename in filenames:\n            save_table(tables[key], p["tables"] / filename)\n\n    summary = execution_summary(cfg, tables, leveled)\n    (p["tables"] / "execution_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")\n    tables["execution_summary"] = pd.DataFrame([summary])\n    return tables\n\n\ndef execution_summary(cfg: PipelineConfig, tables: dict[str, pd.DataFrame], leveled_stations: list[str]) -> dict:\n    p = paths(cfg)\n    return {\n        "generated_at": datetime.now().isoformat(timespec="seconds"),\n        "project_root": str(p["root"]),\n        "date_start": cfg.date_start,\n        "date_end": cfg.date_end,\n        "match_window_days": cfg.match_window_days,\n        "n_metadata_stations": int(len(tables.get("metadata", pd.DataFrame()))),\n        "n_leveled_stations": int(len(leveled_stations)),\n        "n_matchups_raw": int(len(tables.get("matchups_raw", pd.DataFrame()))),\n        "n_matchups_quality_filtered": int(len(tables.get("matchups_quality_filtered", pd.DataFrame()))),\n        "n_matchups_outlier_filtered": int(len(tables.get("matchups_outlier_filtered", pd.DataFrame()))),\n        "n_absolute_matchups": int(len(tables.get("metrics_absolute_by_station", pd.DataFrame()))),\n        "n_dynamic_station_product_series": int(len(tables.get("metrics_dynamic_by_station", pd.DataFrame()))),\n        "tables_dir": str(p["tables"]),\n        "figures_dir": str(p["figures"]),\n    }\n\n\ndef savefig(cfg: PipelineConfig, fig, name: str) -> Path:\n    out = paths(cfg)["figures"] / f"{name}.png"\n    fig.savefig(out, bbox_inches="tight", dpi=300)\n    plt.close(fig)\n    return out\n\n\ndef plot_absolute_product_metrics(cfg: PipelineConfig, metrics: pd.DataFrame) -> Path | None:\n    if metrics.empty:\n        return None\n    fig, axes = plt.subplots(1, 3, figsize=(13, 4))\n    for ax, metric, label in zip(axes, ["mae", "rmse", "bias"], ["MAE (m)", "RMSE (m)", "Bias (m)"]):\n        data = metrics.sort_values(metric)\n        ax.bar(data["product"].map(PRODUCT_LABELS), data[metric], color=[PRODUCT_COLORS.get(p, "#888") for p in data["product"]])\n        ax.axhline(0, color="#333", lw=0.8)\n        ax.set_ylabel(label)\n        ax.grid(axis="y", alpha=0.25)\n    fig.suptitle("Absolute performance by product")\n    fig.tight_layout()\n    return savefig(cfg, fig, "fig_absolute_product_metrics")\n\n\ndef plot_dynamic_summary(cfg: PipelineConfig, metrics: pd.DataFrame) -> Path | None:\n    if metrics.empty:\n        return None\n    fig, ax = plt.subplots(figsize=(7, 4))\n    data = metrics.sort_values("median_spearman", ascending=False)\n    ax.bar(data["product"].map(PRODUCT_LABELS), data["median_spearman"], color=[PRODUCT_COLORS.get(p, "#888") for p in data["product"]])\n    ax.set_ylim(-1, 1)\n    ax.axhline(0, color="#333", lw=0.8)\n    ax.set_ylabel("Median Spearman")\n    ax.set_title("Normalized dynamic validation")\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    return savefig(cfg, fig, "fig_dynamic_spearman_summary")\n\n\ndef plot_station_diagnostic(cfg: PipelineConfig, matchups: pd.DataFrame, station: str | None = None, product: str | None = None) -> Path | None:\n    if matchups.empty:\n        return None\n    work = matchups.copy()\n    station = station or work.groupby("station").size().sort_values(ascending=False).index[0]\n    product = product or work[work["station"] == station].groupby("product").size().sort_values(ascending=False).index[0]\n    group = work[(work["station"] == station) & (work["product"] == product)].sort_values("date_ref")\n    if group.empty:\n        return None\n    met = calc_metrics(group["wse_insitu"], group["wse_sat"], include_bias=True)\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    axes[0].plot(group["date_ref"], group["wse_insitu"], "-o", ms=3, label="In situ", color="#1f77b4")\n    axes[0].plot(group["date_sat"], group["wse_sat"], "-o", ms=3, label=PRODUCT_LABELS.get(product, product), color=PRODUCT_COLORS.get(product, "#888"))\n    axes[0].set_ylabel("WSE (m)")\n    axes[0].legend(frameon=False)\n    axes[0].grid(alpha=0.25)\n    axes[1].scatter(group["wse_insitu"], group["wse_sat"], color=PRODUCT_COLORS.get(product, "#888"), alpha=0.8)\n    lo = np.nanmin([group["wse_insitu"].min(), group["wse_sat"].min()])\n    hi = np.nanmax([group["wse_insitu"].max(), group["wse_sat"].max()])\n    axes[1].plot([lo, hi], [lo, hi], "--", color="#333", lw=1)\n    axes[1].set_xlabel("In situ WSE (m)")\n    axes[1].set_ylabel("Satellite/altimetry WSE (m)")\n    axes[1].set_title(f"MAE={met[\'mae\']:.2f} m | RMSE={met[\'rmse\']:.2f} m | rho={met[\'spearman_r\']:.2f}")\n    axes[1].grid(alpha=0.25)\n    fig.suptitle(f"Station {station} - {PRODUCT_LABELS.get(product, product)}")\n    fig.tight_layout()\n    return savefig(cfg, fig, f"fig_station_diagnostic_{station}_{product}")\n\n\ndef plot_outlier_audit(cfg: PipelineConfig, audit: pd.DataFrame) -> Path | None:\n    if audit.empty or "is_outlier_any" not in audit.columns:\n        return None\n    data = audit.groupby("product", as_index=False).agg(n_total=("is_outlier_any", "size"), n_removed=("is_outlier_any", "sum"))\n    data["removed_pct"] = 100 * data["n_removed"] / data["n_total"].replace(0, np.nan)\n    fig, ax = plt.subplots(figsize=(7, 4))\n    ax.bar(data["product"].map(PRODUCT_LABELS), data["removed_pct"], color=[PRODUCT_COLORS.get(p, "#888") for p in data["product"]])\n    ax.set_ylabel("Outliers removed (%)")\n    ax.set_title("Hampel filter audit")\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    return savefig(cfg, fig, "fig_hampel_outlier_audit")\n\n\ndef generate_figures(cfg: PipelineConfig, tables: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    figure_paths = [\n        plot_absolute_product_metrics(cfg, tables.get("metrics_absolute_overall", pd.DataFrame())),\n        plot_dynamic_summary(cfg, tables.get("metrics_dynamic_overall", pd.DataFrame())),\n        plot_station_diagnostic(cfg, tables.get("matchups_outlier_filtered", pd.DataFrame())),\n        plot_outlier_audit(cfg, tables.get("outlier_filter_audit", pd.DataFrame())),\n    ]\n    manifest = pd.DataFrame({"figure_path": [str(p) for p in figure_paths if p]})\n    manifest.to_csv(paths(cfg)["tables"] / "figure_manifest.csv", index=False)\n    return manifest\n\n\nSWOT_SHORT_NAMES = {\n    "pixc": ["SWOT_L2_HR_PIXC_D", "SWOT_L2_HR_PIXC_2.0"],\n    "raster": ["SWOT_L2_HR_Raster_D", "SWOT_L2_HR_Raster_2.0"],\n    "node": ["SWOT_L2_HR_RiverSP_D", "SWOT_L2_HR_RiverSP_2.0"],\n    "reach": ["SWOT_L2_HR_RiverSP_D", "SWOT_L2_HR_RiverSP_2.0"],\n}\n\n\ndef run_swot_acquisition(cfg: PipelineConfig) -> pd.DataFrame:\n    raise NotImplementedError(\n        "SWOT acquisition is not enabled in this compact final workflow. "\n        "Populate the configured SWOT station folders before running the final analysis."\n    )\n\n\ndef run_pipeline(cfg: PipelineConfig, make_figures: bool = True) -> dict[str, pd.DataFrame]:\n    if cfg.run_altimetry_downloads:\n        consolidate_altimetry(cfg)\n    if cfg.run_swot_downloads:\n        run_swot_acquisition(cfg)\n    tables = run_analysis(cfg)\n    if make_figures:\n        tables["figure_manifest"] = generate_figures(cfg, tables)\n    return tables\n', encoding="utf-8")
    print(f"Created {module_path.resolve()}")
else:
    print(f"Using existing {module_path.resolve()}")


In [ ]:

from pathlib import Path
import json
import sys
import pandas as pd

for candidate in [Path.cwd(), Path.cwd() / "scripts", Path(".").resolve()]:
    if (candidate / "swot_wse_pipeline.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from swot_wse_pipeline import (
    PipelineConfig,
    config_as_json,
    validate_inputs,
    consolidate_altimetry,
    run_pipeline,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

CFG = PipelineConfig(
    project_root=PROJECT_ROOT,
    # Generic public-repository paths. Change these explicitly if your local layout differs.
    in_situ_unleveled_dir=Path("data/in_situ/unleveled"),
    in_situ_gnss_leveled_dir=Path("data/in_situ/gnss_leveled"),
    metadata_file=Path("data/metadata/station_metadata.csv"),
    water_mask_file=Path("data/masks/water_mask.shp"),
    stations_vector_file=Path("data/vectors/gauge_locations.gpkg"),
    altimetry_codes_file=Path("data/altimetry/altimetry_station_codes.csv"),
    swot_station_dir_template="outputs/swot_timeseries/Station_{station}",
    altimetry_station_dir_template="outputs/altimetry_timeseries/Station_{station}",
    tables_dir=Path("outputs/tables"),
    figures_dir=Path("outputs/figures"),
    date_start="2023-03-31",
    date_end="2025-08-01",
    calval_cutoff_date="2023-07-01",
    match_window_days=3,
    station_filter=tuple(),
    run_altimetry_downloads=False,
    run_swot_downloads=False,
    overwrite_outputs=False,
)

print(config_as_json(CFG))


In [ ]:

input_report = validate_inputs(CFG, strict=False)
display(input_report)


## DAHITI/HydroWeb consolidation

Run this cell when new DAHITI/HydroWeb files are available or when `run_altimetry_downloads=True`. With downloads disabled, it only consolidates files from the explicitly configured `altimetry_station_dir_template`.


In [ ]:

altimetry_summary = consolidate_altimetry(CFG)
display(altimetry_summary.head() if not altimetry_summary.empty else altimetry_summary)


## Final analysis

This step writes temporal matchups, Hampel-filter audits, absolute metrics, dynamic metrics, figures, and the execution summary. To reproduce the same results obtained so far, keep the same input files and the same configuration parameters above.


In [ ]:

tables = run_pipeline(CFG, make_figures=True)

for name in [
    "metrics_absolute_overall",
    "metrics_absolute_by_station",
    "metrics_dynamic_overall",
    "figure_manifest",
]:
    print(f"\n{name}")
    display(tables.get(name, pd.DataFrame()).head())


In [ ]:

summary_path = Path(CFG.project_root) / "outputs" / "tables" / "execution_summary.json"
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))


## Reproducibility checks

Main final outputs:

- `outputs/tables/matchups_outlier_filtered.csv`
- `outputs/tables/metrics_overall.csv`
- `outputs/tables/metrics_by_station.csv`
- `outputs/tables/metrics_by_product_orbital_phase.csv`
- `outputs/tables/error_by_hydrological_phase.csv`
- `outputs/tables/metrics_dynamic_by_station.csv`
- `outputs/tables/metrics_dynamic_overall.csv`
- `outputs/figures/*.png`

If results change, compare `outputs/tables/execution_summary.json`, the input files, and the configuration cell first.
